# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [27]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
# crean un mapa de 5 filas por 6 columnas.
        self.height = 5
        self.width = 6
# indica dónde inicia el robot.
        self.start = (4, 0)
# guarda las celdas bloqueadas por estanterías. El robot no puede entrar allí.
        self.walls = {
            (0, 2), (0, 3),
            (1, 1), (1, 3),
            (2, 1), (2, 2),
            (3, 3), (3, 4),
        }
# identifica pisos resbalosos, donde el robot se desvía con más frecuencia.
        self.slippery_states = {
            (1, 2), (2, 0), (3, 1), (3, 2), (2, 4)
        }
# define destinos que terminan el episodio, con sus recompensas: entrega +10, carga +2 y peligro mortal -10.
        self.terminal_states = {
            (0, 5): 10.0,
            (2, 5): 2.0,
            (4, 5): -10.0,
        }
# son peligros de -3, pero el robot puede seguir moviéndose después de caer allí.
        self.danger_states = {
            (1, 4): -3.0,
            (2, 3): -3.0,
            (4, 3): -3.0,
        }
#cada paso normal cuesta un punto; así se favorecen rutas cortas.
        self.living_reward = -1.0
        self.gamma = 0.9
#representa arriba, abajo, izquierda y derecha mediante cambios de fila y columna.
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]
# revisa que una posición esté dentro del mapa y no sea pared.
    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if (r, c) in self.walls:
            return False
        return True
# genera todas las celdas transitables.
    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]
#  comprueba si una celda termina la interacción.
    def is_terminal(self, state):
        return state in self.terminal_states
# devuelve la recompensa de una celda.
    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        if self.is_valid_state(state):
            return self.living_reward
        return 0.0
# calcula posibles resultados de una acción. 
    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if action == (-1, 0):
            left = (0, -1)
            right = (0, 1)
        elif action == (1, 0):
            left = (0, 1)
            right = (0, -1)
        elif action == (0, -1):
            left = (1, 0)
            right = (-1, 0)
        else:
            left = (-1, 0)
            right = (1, 0)

        if state in self.slippery_states:
            outcomes = [(action, 0.60), (left, 0.20), (right, 0.20)]
        else:
            outcomes = [(action, 0.90), (left, 0.05), (right, 0.05)]

        result = {}
        for delta, probability in outcomes:
            next_state = (state[0] + delta[0], state[1] + delta[1])
            if not self.is_valid_state(next_state):
                next_state = state
            result[next_state] = result.get(next_state, 0.0) + probability

        return list(result.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [28]:
# crea el almacén definido antes.
grid = WarehouseMDP()

#  obtiene todos los estados válidos.
S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
#El doble for revisa cada estado y cada acción posible.
for s in S:
    for a in grid.actions:
        # devuelve pares como (siguiente_estado, probabilidad).
        transitions = grid.get_transition_probs(s, a) 
        # suma solo las probabilidades.
        total = sum(p for _, p in transitions) 
        # detiene la ejecución si la suma no es prácticamente 1. 
        assert abs(total - 1.0) < 1e-12 
        # El margen 1e-12 tolera errores minúsculos de decimales en computador.

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 22
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


Implementa Value Iteration para hallar el valor óptimo de cada celda y luego obtiene la acción óptima para cada estado.

In [29]:
"""
Recorre todos los posibles estados siguientes.
Multiplica el valor de cada siguiente estado por su probabilidad.
Los suma para obtener el valor esperado de tomar una acción.
"""

def expected_next_value(grid, state, action, V):
    value = 0.0
    for next_state, probability in grid.get_transition_probs(state, action):
        value += probability * V[next_state]
    return value


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
# Crea V con valor inicial 0 para cada estado.
    V = {s: 0.0 for s in grid.states()}
# Repite el cálculo hasta que los valores casi no cambien.
    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        delta = 0.0

        for s in grid.states():
            # Para estados terminales, fija directamente su recompensa, como +10.
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            # Para los demás estados, prueba las cuatro acciones.
            else:
                best_value = -np.inf
                for action in grid.actions:
                # Calcula q, que es el valor esperado de cada acción
                    q = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
                    if q > best_value:
                        # Conserva el mayor resultado en best_value.
                        best_value = q
                new_V[s] = best_value
    # delta mide cuánto cambió el valor más afectado en una iteración.
            delta = max(delta, abs(new_V[s] - V[s]))

        """ Cuando delta < threshold, considera que 
        ya llegó a una solución suficientemente estable 
        y devuelve los valores junto con el número de iteraciones."""
        V = new_V
        if delta < threshold:
            return V, iteration

    return V, max_iter


def extract_policy(grid, V):
    # Usa los valores finales obtenidos
    policy = {}
    # Para cada estado no terminal, vuelve a probar las acciones.
    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = -np.inf
        for action in grid.actions:
            candidate = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
            if candidate > best_value:
                best_value = candidate
                best_action = action
# Guarda la acción con mayor valor esperado en policy.
        policy[s] = best_action

    return policy


Nota metacognitiva: value_iteration responde “¿qué tan conveniente es cada celda?”, mientras que extract_policy responde “¿hacia dónde debe moverse el robot desde cada celda?”.


## Parte 3 — Policy Iteration

### Policy Evaluation
Calcula qué tan buena es la política actual.
$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement
Cambia las acciones por otras mejores según esos valores.
$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$
Se repiten ambas fases hasta que ninguna acción cambie

In [30]:
# Calcula el valor de cada estado siguiendo una política fija.
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Inicia el valor de cada estado con su recompensa inmediata.
    V = {s: grid.get_reward(s) for s in grid.states()}
    
    for _ in range(max_iter):
        new_V = V.copy()
        delta = 0.0

        for s in grid.states():
            # En estados terminales conserva su recompensa, como +10 o -10.
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                action = policy[s]
                # Suma la recompensa actual y el valor futuro promedio.
                value = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
                new_V[s] = value

            # Delta mide el cambio máximo; cuando es pequeño, la evaluación se detiene.
            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V
        if delta < threshold:
            break

    # Devuelve los valores de los estados para esa política.
    return V


# Construye una política mejor usando los valores calculados en la fase anterior.
def policy_improvement(grid, V, policy=None):
    if policy is None:
        policy = {}

    new_policy = {}
    for s in grid.states():
        # Omite los estados terminales porque allí no se toma ninguna acción.
        if grid.is_terminal(s):
            continue

        # Conserva la acción actual cuando los valores son prácticamente iguales.
        current_action = policy.get(s, grid.actions[0])
        best_action = current_action
        best_value = grid.get_reward(s) + grid.gamma * expected_next_value(
            grid, s, current_action, V
        )

        # Solo cambia la acción si existe una mejora mayor que el error numérico.
        for action in grid.actions:
            candidate = grid.get_reward(s) + grid.gamma * expected_next_value(
                grid, s, action, V
            )
            if candidate > best_value + 1e-12:
                best_value = candidate
                best_action = action

        new_policy[s] = best_action

    return new_policy


# Coordina el algoritmo completo: evalúa y mejora hasta estabilizar la política.
def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Inicializa la política con la primera acción para cada estado no terminal.
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    # Guarda información de cada ronda.
    history = []

    # Repite hasta que la política ya no cambie o se alcance el máximo de iteraciones.
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V, policy)
        history.append(len(new_policy))

        if new_policy == policy:
            # La política ya no cambia, por lo que se detiene la iteración.
            return new_policy, V, history

        policy = new_policy

    return policy, V, history


## Parte 4 — Visualización y comparación


In [31]:
""" Prepara funciones para mostrar en pantalla 
los valores y la política del robot como una cuadrícula legible."""
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}
# Recorre fila por fila y columna por columna.
def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                # Si encuentra una pared, escribe WALL.
                row.append("  WALL  ")
            else:
                # Si es una celda transitable,
                # muestra su valor con tres decimales,
                # por ejemplo +4.281.
                row.append(f"{V[s]:+7.3f}")
        #  Une las celdas de una fila con separadores verticales.
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                # Las paredes se muestran como #.
                row.append(" # ")
            elif grid.is_terminal(s):
                # Los estados terminales muestran su recompensa: +10, +2 o -10.
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                # Los otros estados muestran 
                # la flecha de la acción que indica la política.
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [32]:
# VALUE ITERATION
# Usa una tolerancia estricta para calcular valores precisos.
V_vi, n_vi = value_iteration(grid, threshold=1e-8)
# Extrae una política óptima a partir de los valores calculados.
pi_vi = extract_policy(grid, V_vi)

print("VALUE ITERATION")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
# Usa la misma tolerancia para que ambos métodos se comparen justamente.
pi_pi, V_pi, history = policy_iteration(grid, threshold=1e-8)

print("\nPOLICY ITERATION")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

# Compara los valores: pueden existir políticas óptimas con flechas diferentes.
assert all(abs(V_vi[s] - V_pi[s]) < 1e-3 for s in grid.states())
print("\n✓ Ambos algoritmos encontraron valores óptimos equivalentes.")

VALUE ITERATION
Iteraciones: 176

Valores:
-10.000 | -10.000 |   WALL   |   WALL   |  +7.607 | +10.000
-10.000 |   WALL   | -10.000 |   WALL   |  +3.670 |  +7.607
-10.000 |   WALL   |   WALL   |  -2.493 |  +0.903 |  +2.000
-10.000 | -10.000 | -10.000 |   WALL   |   WALL   |  +0.681
-10.000 | -10.000 | -10.000 | -12.198 | -10.000 | -10.000

Política:
 ↓  |  ←  |  #  |  #  |  →  | +10
 ←  |  #  |  ↑  |  #  |  ↑  |  ↑ 
 ↑  |  #  |  #  |  →  |  →  | +2
 ↓  |  ↑  |  ↑  |  #  |  #  |  ↑ 
 ↑  |  ←  |  ←  |  ←  |  →  | -10

POLICY ITERATION
Historia: [19, 19, 19]

Valores:
-10.000 | -10.000 |   WALL   |   WALL   |  +7.607 | +10.000
-10.000 |   WALL   | -10.000 |   WALL   |  +3.670 |  +7.607
-10.000 |   WALL   |   WALL   |  -2.493 |  +0.903 |  +2.000
-10.000 | -10.000 | -10.000 |   WALL   |   WALL   |  +0.681
-10.000 | -10.000 | -10.000 | -12.198 | -10.000 | -10.000

Política:
 ↑  |  ↑  |  #  |  #  |  →  | +10
 ↑  |  #  |  ↑  |  #  |  ↑  |  ↑ 
 ↑  |  #  |  #  |  →  |  →  | +2
 ↑  |  ↑  |  ↑  | 

## Parte 5 — Interpreta la política

### 1. Desde `START`, ¿el robot busca la entrega `+10` o prefiere la estación de carga `+2`?

**Respuesta:** La política base indica `UP` desde `START`. La decisión óptima no depende solo del premio final: considera la distancia, los riesgos y el costo de cada paso. Los experimentos siguientes permiten observar cómo cambia esa decisión.

### 2. ¿Por qué una recompensa menor podría ser óptima?

**Respuesta:** La carga `+2` puede ser preferible si está más cerca o si el camino hacia `+10` tiene más riesgo de caer en peligros. El agente elige el mayor resultado esperado, no necesariamente la recompensa más grande.

### 3. ¿En qué estados el piso resbaloso cambia la decisión?

**Respuesta:** Principalmente en `(1, 2)`, `(2, 0)`, `(3, 1)`, `(3, 2)` y `(2, 4)`. Allí el robot tiene solo un $60\%$ de avanzar en la dirección elegida, por lo que una ruta cercana a peligros puede dejar de ser atractiva.

### 4. ¿Qué papel cumple el costo por paso `-1`?

**Respuesta:** Penaliza los recorridos largos. Por eso el robot busca llegar pronto a una meta y evita quedarse atrapado, incluso si la entrega más grande está más lejos.

### 5. ¿Por qué $T(s,a,s')$ no puede implementarse con las mismas probabilidades para todos los estados?

**Respuesta:** Porque la probabilidad depende del tipo de piso del estado actual. En piso normal la acción elegida ocurre con $0.90$; en piso resbaloso ocurre con $0.60$. Además, paredes y bordes hacen que el robot permanezca en la misma celda.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

**Predicción:** al reducir el costo de `-1` a `-0.1`, las rutas largas se castigan menos. Por tanto, la entrega `+10` debería resultar más atractiva.

In [33]:
# EXPERIMENTO A: reducir el costo de cada paso.
# Crea un nuevo mundo para no modificar el experimento original.
grid_a = WarehouseMDP()
# Cambia el costo por paso de -1 a -0.1.
grid_a.living_reward = -0.1

# Calcula los valores y una política óptima para el nuevo escenario.
V_a, n_a = value_iteration(grid_a, threshold=1e-8)
pi_a = extract_policy(grid_a, V_a)

print("EXPERIMENTO A: MENOS COSTO POR PASO")
print("Costo por paso:", grid_a.living_reward)
print("Iteraciones:", n_a)
print("Acción desde START:", ARROWS[pi_a[grid_a.start]])
print("\nPolítica:")
print_policy(grid_a, pi_a)

EXPERIMENTO A: MENOS COSTO POR PASO
Costo por paso: -0.1
Iteraciones: 154
Acción desde START: ↑

Política:
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  ↑  |  #  |  ↑  |  ↑ 
 ←  |  #  |  #  |  →  |  ↑  | +2
 ↑  |  ↑  |  ←  |  #  |  #  |  ↑ 
 ↑  |  ↑  |  ←  |  ←  |  ←  | -10


### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

**Predicción:** con solo $40\%$ de éxito en suelo resbaloso, el robot debería evitar más esos estados y preferir rutas seguras.

In [34]:
# EXPERIMENTO B: hacer el piso resbaloso aún más incierto.
class VerySlipperyWarehouseMDP(WarehouseMDP):
    def get_transition_probs(self, state, action):
        # Los estados terminales conservan su transición a sí mismos.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Define las dos desviaciones perpendiculares a la acción elegida.
        if action == (-1, 0):
            left, right = (0, -1), (0, 1)
        elif action == (1, 0):
            left, right = (0, 1), (0, -1)
        elif action == (0, -1):
            left, right = (1, 0), (-1, 0)
        else:
            left, right = (-1, 0), (1, 0)

        # En piso resbaloso: 40% dirección elegida y 30% para cada desvío.
        if state in self.slippery_states:
            outcomes = [(action, 0.40), (left, 0.30), (right, 0.30)]
        else:
            outcomes = [(action, 0.90), (left, 0.05), (right, 0.05)]

        result = {}
        for delta, probability in outcomes:
            next_state = (state[0] + delta[0], state[1] + delta[1])
            # Un borde o pared hace que el robot permanezca en el estado actual.
            if not self.is_valid_state(next_state):
                next_state = state
            result[next_state] = result.get(next_state, 0.0) + probability

        return list(result.items())

# Resuelve el MDP con el nuevo nivel de deslizamiento.
grid_b = VerySlipperyWarehouseMDP()
V_b, n_b = value_iteration(grid_b, threshold=1e-8)
pi_b = extract_policy(grid_b, V_b)

print("EXPERIMENTO B: PISO MUY RESBALOSO")
print("Probabilidad de la dirección elegida en piso resbaloso: 0.40")
print("Iteraciones:", n_b)
print("Acción desde START:", ARROWS[pi_b[grid_b.start]])
print("\nPolítica:")
print_policy(grid_b, pi_b)

EXPERIMENTO B: PISO MUY RESBALOSO
Probabilidad de la dirección elegida en piso resbaloso: 0.40
Iteraciones: 176
Acción desde START: ↑

Política:
 ↓  |  ←  |  #  |  #  |  →  | +10
 ←  |  #  |  ↑  |  #  |  ↑  |  ↑ 
 ↑  |  #  |  #  |  →  |  →  | +2
 ↓  |  ↑  |  ↑  |  #  |  #  |  ↑ 
 ↑  |  ←  |  ←  |  ←  |  →  | -10


### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

**Predicción:** con $\gamma=0.99$, las recompensas futuras pesan más. Por ello, la entrega distante `+10` debería ganar importancia.

In [35]:
# EXPERIMENTO C: aumentar la importancia de las recompensas futuras.
# Crea un nuevo mundo para conservar intacta la configuración original.
grid_c = WarehouseMDP()
# Cambia gamma de 0.9 a 0.99.
grid_c.gamma = 0.99

# Calcula los valores y la política con mayor paciencia del agente.
V_c, n_c = value_iteration(grid_c, threshold=1e-8)
pi_c = extract_policy(grid_c, V_c)

print("EXPERIMENTO C: MAS PACIENCIA")
print("Gamma:", grid_c.gamma)
print("Iteraciones:", n_c)
print("Acción desde START:", ARROWS[pi_c[grid_c.start]])
print("\nPolítica:")
print_policy(grid_c, pi_c)

EXPERIMENTO C: MAS PACIENCIA
Gamma: 0.99
Iteraciones: 1834
Acción desde START: →

Política:
 ↓  |  ←  |  #  |  #  |  →  | +10
 ↓  |  #  |  ↑  |  #  |  ↑  |  ↑ 
 ↓  |  #  |  #  |  →  |  ↑  | +2
 ↓  |  ↓  |  ↓  |  #  |  #  |  ↑ 
 →  |  →  |  →  |  →  |  →  | -10


### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

La celda siguiente explora costos entre `-3.0` y `0.0` en pasos de `0.01` y reporta el punto donde cambia la acción inicial.

In [36]:
# BONUS: busca el costo por paso donde cambia la decisión inicial.
def action_from_start(living_reward):
    # Crea un mundo con el costo por paso que se quiere evaluar.
    candidate_grid = WarehouseMDP()
    candidate_grid.living_reward = living_reward

    # Resuelve el mundo y devuelve la acción recomendada desde START.
    candidate_values, _ = value_iteration(candidate_grid, threshold=1e-6)
    candidate_policy = extract_policy(candidate_grid, candidate_values)
    return candidate_policy[candidate_grid.start]

# Recorre valores desde -3.0 hasta 0.0 con pasos de 0.01.
costs = np.arange(-3.0, 0.01, 0.01)
previous_action = action_from_start(costs[0])
changes = []

for living_reward in costs[1:]:
    current_action = action_from_start(living_reward)
    # Guarda el punto donde la acción inicial cambia.
    if current_action != previous_action:
        changes.append((living_reward, previous_action, current_action))
        previous_action = current_action

print("BONUS: CAMBIO DE DECISION DESDE START")
if changes:
    for living_reward, old_action, new_action in changes:
        print(
            f"Cerca de living_reward = {living_reward:.2f}: "
            f"{ARROWS[old_action]} cambia a {ARROWS[new_action]}"
        )
else:
    print("No hubo cambio de acción inicial en el intervalo evaluado.")

BONUS: CAMBIO DE DECISION DESDE START
Cerca de living_reward = -1.24: → cambia a ↑
